# Short-term volatility using Binance tick data (ETHUSDC, DOGEUSDT)

**Finding.** 
- Expected price move over the ~10 second exclusivity window, from three noise-robust methods reported as a range: **ETHUSDC 1.0-1.1 bps, DOGEUSDT 2.3-2.5 bps** on 3 days of tick data. 
- The two asset pairs differ by ~2-2.5x, so incorporating asset-specific volatility into the penalty cap is important.

**The challenge with short-term volatility estimation :**
- We want to estimate the typical price move over ~10 seconds, per trading pair.
- Prices tend to alternate between the ask and the bid, so the executed price bounces even when the market is not really moving. When we zoom in, the bounce looks like volatility but is not.
- Summing squared tick moves therefore mismeasures volatility: the bounce inflates it (`E[RV] = IV + 2n x noise variance`), while real moves that complete gradually over several ticks get sliced into fragments and undercounted. The balance shifts with the sampling frequency, so there is no single "right" one to choose.

**The proposed approach:**
- Clean the ticks and measure the noise (Section 1).
- Three academically published methods that each remove the bounce differently: TSRV, realized kernel, pre-averaging (Section 2). Each paper's method is followed as published and reported separately; the degree of consensus among the methods is the validation (Section 3).
- Estimate on 5-minute windows, scale to 10s by the square-root-of-time rule (Section 4). Real price jumps stay in the sample, so a jump inside the window is exactly the risk that matters and what we want to account for.

Notes:

- vol is in basis points = `sqrt(IV) x 1e4`, where IV is the total squared price variation of a window; 1 bp = 0.01%. 
- data is obtained fro Binance, daily aggTrades

In [ ]:
import os, io, zipfile, urllib.request, warnings
import numpy as np, pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
warnings.filterwarnings("ignore")

# ---- config ----
SYMBOLS = ["ETHUSDC", "DOGEUSDT"]
DAYS = ["2026-05-01", "2026-05-02", "2026-05-03"]   # daily aggTrades; small + fast
CACHE = "data/binance_ticks"   # ticks ship with the repo; run from the repo root
os.makedirs(CACHE, exist_ok=True)
WINDOW_S = 300          # base estimation window (5 min)
EXCL_S   = 10           # CoW exclusivity window target (~10s)
COLS = ["aggId","price","qty","firstId","lastId","ts","isBuyerMaker","isBestMatch"]

# ---- plot style: plotly rendered static ----
pio.renderers.default = "png"
pio.renderers["png"].scale = 2
C = {"ETHUSDC": "#2a78d6", "DOGEUSDT": "#eb6834"}   # fixed pair colors, never repainted
MC = {"tsrv": ("TSRV", "#2a78d6"), "kernel": ("kernel", "#1baf7a"),
      "preavg": ("pre-averaging", "#4a3aa7")}        # fixed method colors
GRAY, INK, GRID, SURF, AXIS = "#898781", "#0b0b0b", "#e1e0d9", "#fcfcfb", "#c3c2b7"
PT = lambda idx: list(pd.DatetimeIndex(idx).to_pydatetime())

def style(fig, w=1050, h=430, title=None):
    fig.update_layout(width=w, height=h, template="none", paper_bgcolor=SURF, plot_bgcolor=SURF,
                      font=dict(size=13, color=INK), margin=dict(l=65, r=30, t=60, b=50),
                      legend=dict(bgcolor="rgba(0,0,0,0)"))
    if title: fig.update_layout(title=dict(text=title, font=dict(size=15)))
    fig.update_xaxes(gridcolor=GRID, zeroline=False, linecolor=AXIS)
    fig.update_yaxes(gridcolor=GRID, zeroline=False, linecolor=AXIS)
    return fig


In [ ]:
def download_day(symbol, day):
    fn = f"{symbol}-aggTrades-{day}.zip"
    path = os.path.join(CACHE, fn)
    if not os.path.exists(path):
        url = f"https://data.binance.vision/data/spot/daily/aggTrades/{symbol}/{fn}"
        try:
            urllib.request.urlretrieve(url, path)
        except Exception as e:
            print("  MISS", fn, e); return None
    return path

def load_aggtrades(symbol, days):
    frames = []
    for d in days:
        p = download_day(symbol, d)
        if not p: continue
        with zipfile.ZipFile(p) as z:
            name = z.namelist()[0]
            raw = z.open(name).read(64)
            has_header = not raw[:1].isdigit()
            df = pd.read_csv(zipfile.ZipFile(p).open(name),
                             header=0 if has_header else None,
                             names=None if has_header else COLS,
                             usecols=(lambda c: True) if has_header else ["price","qty","ts","isBuyerMaker"])
            if has_header:  # normalize binance header variants
                df.columns = [str(c).lower() for c in df.columns]
                tcol = [c for c in df.columns if "time" in c or c=="ts" or "transact" in c][0]
                pcol = [c for c in df.columns if c=="price"][0]
                df = df.rename(columns={tcol:"ts", pcol:"price"})
            df = df[["price","ts"]].copy()
            df["ts"] = df["ts"].astype("int64")
            unit = "us" if df["ts"].iloc[0] > 10**14 else "ms"   # ms->us cutover 2025-01-01
            df["t"] = pd.to_datetime(df["ts"], unit=unit, utc=True)
            df["price"] = df["price"].astype(float)
            frames.append(df[["t","price"]])
        print(f"  {symbol} {d}: {len(frames[-1]):,} ticks")
    if not frames: return pd.DataFrame(columns=["t","price"])
    return pd.concat(frames, ignore_index=True).sort_values("t").reset_index(drop=True)

RAW = {s: load_aggtrades(s, DAYS) for s in SYMBOLS}
for s in SYMBOLS: print(s, "raw ticks:", f"{len(RAW[s]):,}")

## 1. Data, cleaning, and the noise fingerprint

- Remove outlier trades. Note, the results do not change significantly even if we did not clean the transactions data.
- Keep the real bid-ask bounce: it is information about the noise that needs to be accounted for and removed from volatility estimation.

The figure shows 3 days of prices, and 60 seconds of raw ticks where the bounce and the filter's removals.

In [ ]:
def clean(df, k=25, mad_mult=3.0, floor_ticks=2.0, floor_bps=1.0):
    df = df[(df.price > 0) & df.price.notna()].copy()
    n0 = len(df)
    df = df.drop_duplicates(subset=["t","price"])
    n_dup = n0 - len(df)
    tick = np.median(np.diff(np.unique(np.sort(df.price.values)))) if df.price.nunique() > 5 else df.price.iloc[0]*1e-5
    med = df.price.rolling(2*k+1, center=True, min_periods=k).median()
    mad = (df.price - med).abs().rolling(2*k+1, center=True, min_periods=k).median()
    floor_abs = max(floor_ticks*tick, df.price.median()*floor_bps*1e-4)
    keep = (df.price - med).abs() <= (mad_mult*mad + floor_abs)
    n_filt = int((~keep.fillna(True)).sum())
    out = df[keep.fillna(True)].reset_index(drop=True).set_index("t")
    return out, tick, n_dup, n_filt

CLEAN = {}
for s in SYMBOLS:
    c, tick, n_dup, n_filt = clean(RAW[s])
    CLEAN[s] = c
    lp = np.log(c.price.values)
    r = np.diff(lp)
    rho1 = np.corrcoef(r[1:], r[:-1])[0,1] if len(r) > 2 else np.nan
    cov1 = np.cov(r[1:], r[:-1])[0,1]
    roll_spread_bps = (2*np.sqrt(-cov1)*1e4) if cov1 < 0 else np.nan
    print(f"{s}: kept {len(c):,} | duplicates {n_dup:,} | filtered {n_filt:,} | tick~{tick:.2e} | "
          f"px~{c.price.median():.4g} | rel-tick {tick/c.price.median()*1e4:.2f} bps | "
          f"lag1 autocorr {rho1:+.3f} | Roll spread ~{roll_spread_bps:.1f} bps")

In [ ]:
# Left: full 3 days at 1s resolution. Right: 60s of raw ticks, filter drops marked x.
ZOOM = {}
fig = make_subplots(rows=2, cols=2, column_widths=[0.62,0.38], horizontal_spacing=0.08,
                    vertical_spacing=0.16,
                    subplot_titles=[t for s in SYMBOLS for t in
                                    (f"{s}: 3 days (1s bars)", f"{s}: 60 seconds of raw ticks")])
for i, s in enumerate(SYMBOLS):
    c = CLEAN[s]
    p1 = c.price.resample("1s").last().ffill()
    fig.add_trace(go.Scatter(x=PT(p1.index), y=p1.values, mode="lines",
                             line=dict(color=C[s], width=0.8), showlegend=False), row=i+1, col=1)
    z0 = c.index[0].floor("h") + pd.Timedelta("90min"); z1 = z0 + pd.Timedelta("60s")
    ZOOM[s] = z0
    zc = c.loc[z0:z1]
    zr = RAW[s][(RAW[s].t >= z0) & (RAW[s].t <= z1)]
    m = zr.merge(zc.reset_index()[["t","price"]].drop_duplicates().assign(_k=1),
                 on=["t","price"], how="left")
    dr = m[m._k.isna()]
    fig.add_trace(go.Scatter(x=PT(zc.index), y=zc.price.values, mode="lines+markers",
                             line=dict(color=C[s], width=1, shape="hv"), marker=dict(size=3),
                             showlegend=False), row=i+1, col=2)
    if len(dr):
        fig.add_trace(go.Scatter(x=PT(dr.t), y=dr.price.values, mode="markers",
                                 marker=dict(symbol="x", size=8, color="#e34948"),
                                 showlegend=False), row=i+1, col=2)
    fig.update_yaxes(title_text="price", row=i+1, col=1)
style(fig, h=560)
fig.show()
print("Right panels: the price ping-pongs between two rails (bid/ask) around a slowly moving level.")

## 2. Methods for short-term volatility estimation from 3 academic papers

Each method returns the total price variation (IV) for one window; each removes the bounce in a different way.
- **naive RV** (`rv_all`): just add up squared moves. It is correct if there is no noise, but wrong with noise. It's the baseline to beat (but not source of truth).
- **`noise_var`**: the size of the bounce itself (`tick RV / 2n`).
- **TSRV** (Zhang, Mykland and Ait-Sahalia 2005): based on 2-samples measurements. Averaging over sparse every-K-th-tick grids gives a number that is mostly real movement. Subtracting the right amount of the first from the second cancels the bounce. `K` is set so each sparse grid samples roughly every 10s.
- **Parzen realized kernel** (Barndorff-Nielsen, Hansen, Lunde and Shephard): the bounce makes neighbouring moves partly cancel each other. Instead of ignoring that, add those cross-products back with declining (Parzen) weights so the bounce nets out.
- **pre-averaging** (Jacod, Podolskij and Vetter): smooth the price first by averaging over small overlapping blocks; the bounce (which hovers around zero) averages away while the real path survives; then measure on the smoothed series and subtract the small leftover bias.

Each paper's estimator is followed as published, and the three are compared side by side.
The figure below shows each method acting on real DOGEUSDT ticks.

In [ ]:
def rv_all(logp):
    r = np.diff(logp); return float(r @ r)

def noise_var(logp):
    n = len(logp)-1
    return rv_all(logp)/(2*n) if n>0 else np.nan

def tsrv(logp, K):
    # Two-scale RV (Zhang-Mykland-Ait-Sahalia 2005), finite-sample adjusted
    n = len(logp)-1
    if n < 2*K+2: return np.nan
    rva = np.mean([np.sum(np.diff(logp[k::K])**2) for k in range(K)])
    nbar = (n-K+1)/K
    iv = (rva - (nbar/n)*rv_all(logp))/(1-nbar/n)
    return max(iv, 0.0)

def _parzen(x):
    w = np.zeros_like(x, dtype=float)
    m1 = (x>=0)&(x<=0.5); m2 = (x>0.5)&(x<=1)
    w[m1] = 1-6*x[m1]**2+6*x[m1]**3
    w[m2] = 2*(1-x[m2])**3
    return w

def realized_kernel(logp, H, jitter=2):
    # Parzen realized kernel (BNHLS form), endpoint-jittered, non-negative
    lp = logp.astype(float).copy()
    if jitter and len(lp) > 2*jitter+2:
        lp[0]=lp[:jitter].mean(); lp[-1]=lp[-jitter:].mean()
    x = np.diff(lp); n = len(x)
    H = int(min(max(H,1), n//2))
    rk = x @ x
    for h in range(1, H+1):
        w = _parzen(np.array([h/H]))[0]
        if w>0: rk += w*2.0*(x[h:] @ x[:-h])
    return max(rk, 0.0)

def preavg_rv(logp, theta=0.5):
    # Pre-averaged RV (Jacod-Podolskij-Vetter), triangular weights.
    # Finite-sample psi constants, full noise correction psi1/(theta^2 psi2) * omega^2
    # and n/(n-kn+2) edge factor per Christensen-Kinnebrock-Podolskij.
    n = len(logp)-1
    if n < 50: return np.nan
    kn = int(np.floor(theta*np.sqrt(n))); kn = max(2, kn-(kn%2))
    j = np.arange(1, kn); g = np.minimum(j/kn, 1-j/kn)
    dg = np.diff(np.concatenate(([0.0], g, [0.0])))
    psi1 = kn*np.sum(dg**2)
    psi2 = np.sum(g**2)/kn
    dY = np.diff(logp)
    Ybar = np.correlate(dY, g, mode="valid")
    om2 = (dY @ dY)/(2*n)
    iv = (n/(n-kn+2))*(Ybar @ Ybar)/(kn*psi2) - (psi1/(theta**2*psi2))*om2
    return max(iv, 0.0)

def grid_logreturns(win_df, freq="1s"):
    p = win_df.price.resample(freq).last().ffill()
    return np.diff(np.log(p.values)) if len(p)>1 else np.array([])
print("estimators defined")

In [ ]:
# What each method does, on 90s of DOGEUSDT ticks.
s = "DOGEUSDT"
seg = CLEAN[s].loc[ZOOM[s] : ZOOM[s] + pd.Timedelta("90s")]
fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.09,
                    subplot_titles=("TSRV: sparse subgrids skip the bounce",
                                    "Kernel: RV before vs after correction",
                                    "Pre-averaging: local smoothing"))
# (a) TSRV: show 3 of K subgrids
fig.add_trace(go.Scatter(x=PT(seg.index), y=seg.price.values, mode="lines",
                         line=dict(color=AXIS, width=1), name="all ticks"), row=1, col=1)
Kd = 8
for k, col in zip(range(3), ["#2a78d6", "#1baf7a", "#4a3aa7"]):
    fig.add_trace(go.Scatter(x=PT(seg.index[k::Kd]), y=seg.price.values[k::Kd],
                             mode="lines+markers", line=dict(color=col, width=1),
                             marker=dict(size=4),
                             name=f"subgrid {k+1}"), row=1, col=1)
# (b) kernel: the estimate before vs after the Parzen-weighted correction,
# median across all 5-min windows (a single window is too noisy for this comparison)
_t0 = CLEAN[s].index.min().ceil("min"); _t1 = CLEAN[s].index.max().floor("min")
_edges = pd.date_range(_t0, _t1, freq=f"{WINDOW_S}s")
_naive, _kern = [], []
for _a, _b in zip(_edges[:-1], _edges[1:]):
    _w = CLEAN[s].loc[_a:_b]
    if len(_w) < 200: continue
    _lp = np.log(_w.price.values)
    _naive.append(rv_all(_lp))
    _kern.append(realized_kernel(_lp, H=max(2, int(np.ceil(len(_lp)**0.6/4)))))
naive_bps = np.sqrt(np.median(_naive))*1e4
kern_bps = np.sqrt(np.median(_kern))*1e4
fig.add_trace(go.Bar(x=["naive RV<br>(every tick)", "kernel<br>estimate"],
                     y=[naive_bps, kern_bps], marker_color=[GRAY, "#1baf7a"], width=0.55,
                     text=[f"{naive_bps:.1f}", f"{kern_bps:.1f}"], textposition="outside",
                     cliponaxis=False, showlegend=False), row=1, col=2)
fig.add_annotation(text=f"correction: {kern_bps-naive_bps:+.1f} bps",
                   x=0.5, y=max(naive_bps, kern_bps)*1.18, showarrow=False,
                   font=dict(color=INK, size=12), row=1, col=2)
fig.update_yaxes(title_text="median 5-min window vol (bps)",
                 range=[0, max(naive_bps, kern_bps)*1.35], row=1, col=2)
fig.update_yaxes(title_text="price", row=1, col=1)
fig.update_yaxes(title_text="price", row=1, col=3)
# (c) pre-averaging: triangular local mean over kn ticks (kn=12 for visibility)
kn = 12
wts = np.minimum(np.arange(1, kn+1), np.arange(kn, 0, -1)).astype(float); wts /= wts.sum()
sm = np.convolve(seg.price.values, wts, mode="valid")
xs = seg.index[kn//2 : kn//2 + len(sm)]
fig.add_trace(go.Scatter(x=PT(seg.index), y=seg.price.values, mode="markers",
                         marker=dict(size=2.5, color=GRAY), showlegend=False), row=1, col=3)
fig.add_trace(go.Scatter(x=PT(xs), y=sm, mode="lines",
                         line=dict(color=C[s], width=2), showlegend=False), row=1, col=3)
style(fig, h=430)
fig.update_layout(margin=dict(l=95, t=60, b=95),
                  legend=dict(orientation="h", x=0, xanchor="left", y=-0.30, yanchor="top",
                              font=dict(size=11), itemwidth=30))
fig.show()


**Reading the three panels:**
- **TSRV (left):** gray = every cleaned tick, bounce included. Blue/green/violet = 3 of TSRV's `K` sparse subgrids: the same data sampled every 8th tick, each with a different starting offset (the real estimation uses `K`~10-12, one tick every ~10s; only 3 are drawn to keep the panel readable). Each subgrid steps over the bounce, so its moves are mostly real price changes, yet together the subgrids still use nearly every tick. TSRV averages the `K` subgrid RVs and subtracts the scaled full-grid (gray) RV to cancel the noise that remains.
- **Kernel (middle):** unlike the other two, the kernel corrects the variance directly rather than the price, so there is no smoothed path to draw. It adds the short-lag autocovariances of returns back into RV with declining Parzen weights, which fixes two opposite distortions at once: the bounce (up-down reversals that inflate naive RV) and gradual price adjustment (a real move completing over several ticks gets sliced into fragments and undercounted). The bars compare the median across all 5-minute windows before and after the correction. On a wide-spread pair the bounce dominates and the correction is downward; on DOGEUSDT the gradual-adjustment effect dominates, so the net correction is upward, the same scale-dependence flagged in Section 3.
- **Pre-averaging (right):** a triangular average over `kn` = 12 ticks kills the bounce and keeps the path; RV is then computed on the smoothed returns.

## 3. Window-by-window estimates

- Cut each pair's 3 days into non-overlapping 5-minute windows (those with under 200 trades are skipped) and run every estimator on every window; per-pair medians are printed in bps.
- The trust check: TSRV, kernel and pre-averaging landing on top of each other, window by window, in the figure. Naive RV sitting apart is the microstructure distortion showing up.
- `noise-to-signal` = `sqrt(noise_var / IV)`: small means the windows are dominated by real movement, not bounce.
- One robustness caveat: each estimator has a dial that sets how much averaging is applied, and sweeping the dials (K, H, theta; not shown) shifts the estimates up by roughly 10-30% before they plateau (ETHUSDC toward the high end of that range, DOGEUSDT the low end). Read that as the systematic band around the numbers below.

In [ ]:
def analyze(symbol):
    df = CLEAN[symbol]
    t0 = df.index.min().ceil("min"); t1 = df.index.max().floor("min")
    edges = pd.date_range(t0, t1, freq=f"{WINDOW_S}s")
    mean_dt = np.mean(np.diff(df.index.values).astype("timedelta64[ns]").astype(float))/1e9
    recs=[]
    for a,b in zip(edges[:-1], edges[1:]):
        w = df.loc[a:b]
        if len(w) < 200: continue
        lp = np.log(w.price.values)
        # TSRV subgrids so each subgrid samples ~ every 10s within THIS window
        Kw = int(round(len(w)*10.0/WINDOW_S)); Kw = max(2, min(Kw, (len(w)-2)//2))
        r1 = grid_logreturns(w, "1s")
        nv = noise_var(lp)
        iv_tsrv = tsrv(lp, Kw)
        recs.append(dict(
            t=b, n=len(w), K=Kw,
            rv1s   = (r1@r1) if len(r1)>1 else np.nan,
            tsrv   = iv_tsrv,
            kernel = realized_kernel(lp, H=max(2,int(np.ceil(len(lp)**0.6/4)))),
            preavg = preavg_rv(lp),
            noise_var = nv,
            nts    = np.sqrt(nv/iv_tsrv) if iv_tsrv>0 else np.nan,
        ))
    R = pd.DataFrame(recs)
    return R, int(R.K.median()), mean_dt

RES={}
for s in SYMBOLS:
    R,K,mean_dt = analyze(s); RES[s]=(R,K,mean_dt)
    to_bps = lambda col: np.sqrt(R[col].median())*1e4
    print(f"\n=== {s} === windows={len(R)}  mean tick spacing~{mean_dt:.2f}s  TSRV K(median)={K}")
    print(f"  per-{WINDOW_S}s vol (bps): RV(1s)={to_bps('rv1s'):.1f}  TSRV={to_bps('tsrv'):.1f}  "
          f"kernel={to_bps('kernel'):.1f}  preavg={to_bps('preavg'):.1f}")
    print(f"  noise-to-signal (median): {R.nts.median():.2f}")

In [ ]:
# Naive RV vs the three noise-robust methods, window by window.
fig = make_subplots(rows=2, cols=1, vertical_spacing=0.14,
                    subplot_titles=[f"{s}: per-window vol (bps)" for s in SYMBOLS])
for i, s in enumerate(SYMBOLS):
    R,_,_ = RES[s]
    t = PT(R.t)
    fig.add_trace(go.Scatter(x=t, y=(np.sqrt(R.rv1s)*1e4).values, mode="lines",
                             line=dict(color=GRAY, width=0.8), name="naive RV (1s)",
                             showlegend=(i==0)), row=i+1, col=1)
    for m, (lbl, col) in MC.items():
        fig.add_trace(go.Scatter(x=t, y=(np.sqrt(R[m])*1e4).values, mode="lines",
                                 line=dict(color=col, width=1.0), name=lbl,
                                 showlegend=(i==0)), row=i+1, col=1)
    fig.update_yaxes(title_text="bps", row=i+1, col=1)
style(fig, h=580)
fig.update_layout(legend=dict(orientation="h", y=1.12, x=0))
fig.show()
print("The three noise-robust lines sit on top of each other: that window-level agreement is the validation.")

## 4. Scaling to the 10-second window

- I estimate on stable 5-minute windows and scale down to 10s, because 10 seconds of raw ticks is too noise-dominated to measure directly (mostly noise).

In [ ]:
horizons = np.array([1,2,5,10,15,30,60,120,300,600])
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.09, subplot_titles=SYMBOLS)
rows=[]
for i, s in enumerate(SYMBOLS):
    R,_,_ = RES[s]
    v10s = {}
    for m, (lbl, col) in MC.items():
        sig2 = R[m].median()/WINDOW_S
        fig.add_trace(go.Scatter(x=horizons, y=np.sqrt(sig2*horizons)*1e4, mode="lines+markers",
                                 line=dict(color=col, width=2), marker=dict(size=5),
                                 name=lbl, showlegend=(i==0)), row=1, col=i+1)
        v10s[m] = float(np.sqrt(sig2*EXCL_S)*1e4)
        rows.append(dict(pair=s, method=m, per_sec_vol_bps=np.sqrt(sig2)*1e4,
                         vol_10s_bps=v10s[m], ann_vol_pct=np.sqrt(sig2*365*24*3600)*100))
    fig.add_vline(x=EXCL_S, line=dict(color=GRAY, dash="dash", width=1), row=1, col=i+1)
    fig.add_annotation(text=f"{EXCL_S}s: {min(v10s.values()):.1f}-{max(v10s.values()):.1f} bps",
                       x=np.log10(EXCL_S), y=max(v10s.values()), xanchor="right",
                       yanchor="bottom", ax=-30, ay=-22, showarrow=True, arrowcolor=GRAY,
                       font=dict(size=12), row=1, col=i+1)
    fig.update_xaxes(type="log", title_text="horizon (s)",
                     tickvals=[1,2,5,10,30,60,120,300,600],
                     ticktext=["1","2","5","10","30","60","120","300","600"], row=1, col=i+1)
fig.update_yaxes(title_text="expected move, vol (bps)", row=1, col=1)
style(fig, h=430)
fig.update_layout(legend=dict(x=0.02, y=0.98))
fig.show()

L = pd.DataFrame(rows)
S = L.pivot(index="pair", columns="method", values="vol_10s_bps")[list(MC)].round(3)
S["method_spread"] = (S[list(MC)].max(axis=1) - S[list(MC)].min(axis=1)).round(3)
print(f"vol_{EXCL_S}s by method (bps):")
print(S.to_string())
print("\nfull per-method summary:")
print(L.set_index(["pair","method"]).round(2).to_string())
lo = {s: S.loc[s, list(MC)].min() for s in SYMBOLS}
hi = {s: S.loc[s, list(MC)].max() for s in SYMBOLS}
print(f"\nExpected {EXCL_S}s move across methods -> ETHUSDC {lo['ETHUSDC']:.1f}-{hi['ETHUSDC']:.1f} bps, "
      f"DOGEUSDT {lo['DOGEUSDT']:.1f}-{hi['DOGEUSDT']:.1f} bps.")
print("This is the short-term volatility input for the cap model.")

## 5. Conclusion

- `vol_10s` across the three methods: ETHUSDC ~1.0-1.1 bps, DOGEUSDT ~2.3-2.5 bps, a roughly 2.2-2.3x gap between the 2 pairs over the same days. A single flat per-chain number mis-prices this; this supports the idea that the cap needs a volatility input.
- The three methods agree window by window to a fraction of a bp, so any of them can serve as the volatility estimation method
- The model/estimator uncertainty that remains is roughly 10-30% due to changing of the estimator parameters (how much averaging is applied).